# Bypass Persistent Reuse

Compare four runs with the same numerical parameters: execute once, reuse the whole chain, force one middle step, then force everything. Cache identity includes qualified software, scientific parameters and exact accepted input producers. A fresh execution can therefore invalidate downstream reuse even if its artifact bytes match.

## Prepare the checkout's canonical worker

Run this notebook with the checkout's Pixi `dev` kernel. The worker uses the same
`pyproject.toml` and committed `pixi.lock`, including the installed Artisan source
and declared development dependencies. Preparation captures bounded package
inputs; it does not copy `.git`, `.pixi`, or tutorial output directories.

The next cell prepares that worker if needed. First preparation may download
locked packages and needs disk space for a separate fixed-path environment.
Later operations and pipelines reuse it. Use the same source snapshot for the
controller and worker; changed package inputs require preparation again.
This setup does not provision models or GPUs. Submission only looks up an
existing prepared target.

`LocalRunner()` without a canonical environment merely observes the current
interpreter and is deliberately not cacheable. The notebook workspace is
temporary; use a persistent base directory when keeping runs.

In [ ]:
from __future__ import annotations

from functools import partial
from tempfile import TemporaryDirectory

from artisan.operations.examples import DataGenerator, DataTransformer, MetricCalculator
from artisan.orchestration import PipelineManager, StepDisposition
from artisan.orchestration.runners import LocalRunner
from artisan.schemas import Environment
from artisan.utils import find_project_root, tutorial_setup
from artisan.visualization import inspect_pipeline

workspace = TemporaryDirectory(prefix="artisan-cache-tutorial-")
tutorial_setup = partial(tutorial_setup, base_dir=workspace.name)
env = tutorial_setup("cache")
project_root = find_project_root().resolve()
runner = LocalRunner(
    environment=Environment.from_pixi(project_root / "pyproject.toml", name="dev")
)
# These operations share the worker environment; prepare it once.
runner.prepare(DataGenerator)

In [ ]:
def run_chain(name, *, force_middle=False, force_all=False):
    with PipelineManager.create(
        name=name,
        delta_root=env.delta_root,
        staging_root=env.staging_root,
        working_root=env.working_root,
        default_step_runner=runner,
        skip_cache=force_all,
    ) as pipeline:
        pipeline.run(DataGenerator, name="generate", params={"count": 2, "seed": 42})
        pipeline.run(
            DataTransformer,
            name="transform",
            params={"seed": 100},
            inputs={"dataset": pipeline.output("generate", "datasets")},
            skip_cache=force_middle or force_all,
        )
        pipeline.run(
            MetricCalculator,
            name="score",
            inputs={"dataset": pipeline.output("transform", "dataset")},
        )
        summary = pipeline.finalize()
    assert summary["overall_success"], summary
    return pipeline


baseline = run_chain("baseline")
assert [step.disposition for step in baseline] == [StepDisposition.EXECUTED] * 3

## Reuse unchanged accepted inputs

A hit keeps the original accepted producer. This lets unchanged downstream work reuse the same input authority.

In [ ]:
cached = run_chain("cached")
assert [step.disposition for step in cached] == [StepDisposition.CACHE_HIT] * 3
inspect_pipeline(env.delta_root, pipeline_run_id=cached.config.pipeline_run_id)

## Force the middle step

The generator still reuses its original result. Forcing the transformation creates a new accepted producer; the metric step executes against that new occurrence. Matching CSV bytes alone do not make the producers interchangeable.

In [ ]:
changed = run_chain("force_middle", force_middle=True)
assert [step.disposition for step in changed] == [
    StepDisposition.CACHE_HIT,
    StepDisposition.EXECUTED,
    StepDisposition.EXECUTED,
]
inspect_pipeline(env.delta_root, pipeline_run_id=changed.config.pipeline_run_id)

## Force the whole run

Use the pipeline default when you want every submitted boundary to execute. This does not change a previous immutable selection or erase its evidence.

In [ ]:
fresh = run_chain("force_all", force_all=True)
assert [step.disposition for step in fresh] == [StepDisposition.EXECUTED] * 3
inspect_pipeline(env.delta_root, pipeline_run_id=fresh.config.pipeline_run_id)